# Task 2 — Prompting LLMs

In the first two activities, we saw how we can build automated rule-based and embedding-based systems from scratch for a classification task.

In this notebook, we'll look at how we can use large, pre-trained models for our own classification tasks with just a little bit of prompting. We'll see a couple of different prompting methods:

- Simple prompting techniques with a Q&A style
- More structured prompts, with guided outputs

---

### How to use this notebook
- Run each grey **code cell** by clicking it and pressing **▶** (or `Shift + Enter`).
- Run the cells **in order, top to bottom.**
- You don't need to understand every line. The parts you'll actually change are clearly marked with
  `# 👉 CHANGE THIS`.
- If you see a red error, try to re-run the cell above it, then try again.

We can now load a chat model. In this demo we're going to use TinyLLama as it will load quickly and run within MyBinder. For better performance, you can experiment with running larger models (but these might not run in MyBinder).

In [ ]:
from transformers import pipeline

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
pipe = pipeline("text-generation", model_name, device_map="auto")

## Film Reviews

Let's get our LLM to do the same text classification task we have been looking at all session.

We can write a prompt which describes our problem and how we want the model to answer:

In [ ]:
MAIN_PROMPT = """\
You are a smart and intelligent text classification system.

You are designed to classify film reviews into one of three classes:
- "POSITIVE" - all reviews that largely thought the film was good and would recommend it.
- "NEGATIVE" - all reviews that largely thought the film was bad and would not recommend it.
- "UNSURE" - all reviews that are a mix of both good and bad thoughts.

Output Format:
{'LABEL': 'POSITIVE', 'NEGATIVE', or 'UNSURE'}

If the review does not fall into one of these categories, leave LABEL blank.
]""" # 👉 CHANGE THIS

Then let's turn this into a sequence of messages.

- We'll include the prompt we defined above.
- We then include a few example input and output pairs.
- Finally, we provide the input we wish the model to solve.

In [ ]:
input_sentence = "Label this according to your prompt: The film was fab!" # 👉 CHANGE THIS

messages = [
     {"role": "system", "content": MAIN_PROMPT},
     {"role": "user", "content": "A brilliant film, the acting was superb."},
     {"role": "assistant", "content": '{"LABEL": "POSITIVE"]}'},
     {"role": "user", "content": "A boring, lazy film. Don't bother watching it!"},
     {"role": "assistant", "content": '{"LABEL": "NEGATIVE"}'},
     {"role": "user", "content": input_sentence}
]

output = pipe(messages)[0]["generated_text"][-1]
print(output["content"])

Because the model has output in a standard JSON format, we can parse the result as data:

In [ ]:
import json
output_data = json.loads(output["content"])

print("Label", output_data["LABEL"])

These could then be saved to a file or visualized, eg:

Try changing the sentence and prompt and see what the model produces. As in the previus notebook, try to:

1. Find a sentence where the two models **disagree**.
2. Find a sentence that **fools the AI** (makes it give the wrong answer).
3. Try sarcasm, slang, or emoji. What breaks it?

## Under the hood

The above example used the [chat templating feature](https://huggingface.co/docs/transformers/main/en/chat_templating) of the transformers library.

Behind-the-scenes this is turned into a single long input for the model which includes special tokens indicating who is "speaking" in the chat dialogue.

For example, the sequence of messages:

In [ ]:
messages = [
   {"role": "system", "content": "You are a helpful chatbot."},
   {"role": "user", "content": "Hello, how are you?"},
   {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
   {"role": "user", "content": "I'd like to show off how chat templating works!"},
]

will be turned into the following prompt under-the-hood:

In [ ]:
print(pipe.tokenizer.apply_chat_template(messages, tokenize=False))

Notice how the special tokens `<|system|>`, `<|user|>`, `<|assistant|>`, and `</s>` are added between each round of dialogue.

Each LLM (Large Language Model) is trained using different formats so these special tokens are model-specific. The chat templating feature hides this away for us so we don't have to remember which tokens to use.

We can also see how we transform our text into numbers:

In [ ]:
pipe.tokenizer.encode("This is some test text")

You can see if we look at a different sentence that the same words always get given the same number (e.g., "text" goes to 1426).


In [ ]:
pipe.tokenizer.encode("Isn't it great that I can input text.")

Some words are too complex for the text encoding to handle, so we break them down into `tokens`:

In [ ]:
word_to_encode = "biology"
pipe.tokenizer.encode(word_to_encode)

This is similar to how a human might break down words they haven't encountered before:

In [ ]:
for token in pipe.tokenizer.encode(word_to_encode):
  word = pipe.tokenizer.decode(token)
  print(f"{token} maps to: {word}")

If you like, change the `word_to_encode` above and see how different words or sentences are tokenised.

When these LLMs are trained, they learn how to embed these single numbers into their own embeddings, like we saw with `word2vec` earlier.

# Classification

Lets try another example, this time predicting the sex of the animal as one of 3 categories:

*   Male
*   Female
*   Unknown

In [ ]:
test_sentences = [
    {
        "sentence": "The Owner brought his dog into the surgery yesterday and mentioned a history with diabetes.",
        "label": "unknown"
    },
    {
        "sentence": "I saw a 5yo cat with a broken leg. She didn't show any improvement since her last visit.",
        "label": "female"
    },
    {
        "sentence": "9yo F/N cat",
        "label": "female"
    },
      {
        "sentence": "Shorthair was brought into my clinic the other day, and is the most beautiful boy!",
        "label": "male"
    },
] # 👉 CHANGE THIS

Write a prompt which can classify the sex of the animal (not the owner!):




In [ ]:
MAIN_PROMPT = """\

You are a smart sex classifying vet. Classify the cat in the following text as male, female, or unknown.
Reply with only one of these words: Classify the following text as male, female, or unknown.

""" # 👉 CHANGE THIS

def classify_sex(sentence):
  messages = [
    {"role": "system", "content": MAIN_PROMPT},
    {"role": "user", "content": f"Classify this sentence: {sentence}"}
  ]
  return pipe(messages)[0]["generated_text"][-1]["content"].lower()

In [ ]:
print(classify_sex("I have been working with a 10-year-old diabetic cat. He is treated with 3 units of ProZinc insulin."))

Try it out on the examples:

In [ ]:
for sentence in test_sentences:
  output = classify_sex(sentence["sentence"])
  print("Input:           ", sentence["sentence"])
  print("Label:           ", sentence["label"])
  print("Model Prediction:", output.replace("\n", " \\n"))
  prediction = "female" if "female" in output else "male" if "male" in output else "unknown"
  #print("Verdict          :", "✅" if prediction == sentence["label"].lower() else "❌")
  print()

Did the model get them all right? Prompting is hard and small changes make a big difference to the output. Try modifying your phrasing or adding more examples.

# Guiding the model

Previously, we told the model which format to output the result in and gave it some examples. You may have noticed, it can be quite hard to get the model to follow the format you want!

Most of the time this is enough, but if you're asking the LLM to solve a task it's not seen before (or using a very small model in our case), it may struggle with the output format. This is a problem if we want to parse the model's output and we're expecting it to be in a specific format.

We can ensure that the model outputs in the correct format by using constrained generation. There are loads of libraries which do this, but we'll explore using the [guidance](https://guidance.readthedocs.io/en/latest/) library here.  

Firstly we make sure we have all the dependencies and load the library and a model. We're using the LLM defined in the previous section but you could use other things here including the OpenAI api.

In [ ]:
from guidance import models, select
from transformers import pipeline

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lm = models.Transformers(model_name)

Lets try applying this to classifying the sex of an animal in the input.

We can now define the rules for the output. In this example we want the model to only output one of 3 options:


*   Male
*   Female
*   Unknown

In [ ]:
def classify_sex(input_sentence):
  messages = [
    {"role": "system", "content": MAIN_PROMPT},
    {"role": "user", "content": f"Sentence: He was a happy dog"},
    {"role": "assistant", "content": f"male"},
    {"role": "user", "content": f"Sentence: I met Lucy, a 3yo female shorthair today."},
    {"role": "assistant", "content": f"female"},
    {"role": "user", "content": f"Sentence: A 5yo poodle was brought into the office today by the owner. She described how they wouldn't eat food."},
    {"role": "assistant", "content": f"unknown"},
    {"role": "user", "content": f"Sentence: {sentence}"}
  ]
  prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  prompt = prompt.replace("</s>", "")

  # This line is the magic!
  # Notice how select is given 3 options constraining the output!
  return str(lm + prompt + select(['male', 'female', 'unknown'], name="sex")).split("<|assistant|>")[-1].strip()

In [ ]:
classify_sex("I have been working with a 10-year-old diabetic cat. He is treated with 3 units of ProZinc insulin.") # 👉 CHANGE THIS

In [ ]:
for sentence in test_sentences:
  output = classify_sex(sentence["sentence"])
  print("Input:           ", sentence["sentence"])
  print("Label:           ", sentence["label"])
  print("Model Prediction:", output.replace("\n", " \\n"))
  prediction = "female" if "female" in output else "male" if "male" in output else "unknown"
  print("Verdict          :", "✅" if prediction == sentence["label"].lower() else "❌")
  print()

# Exercises

1. Experiment with different prompts. The structure of the prompt makes a big difference to the performance of the model
2. Explore some of the other things the [guidance library](https://github.com/guidance-ai/guidance/tree/main) can do.